In [17]:
import glob
import os
import shutil
import json

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

from tflite_inference import TFLiteInferer

In [2]:
# !pip install tensorflow==2.3

In [3]:
CONF_TH = 0.35
IOU_TH = 0.5

ORIGINAL_H, ORIGIAL_W = 480, 640
W = H = 320
RATIO = W / ORIGIAL_W

inferer = TFLiteInferer(
    yolo_pth="results/filtered_v3_margin_15/weights/best_saved_model/best_full_integer_quant.tflite",
    head_pth="results/filtered_v3_margin_15/weights/head_saved_model/head_float32.tflite",
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [4]:
def draw(img, detections):
    tmp = img.copy()
    for box in detections:
        cv2.rectangle(tmp, (box[:2] / RATIO).astype(int), ((box[:2].astype(int) + box[2:4]) / RATIO).astype(int) , (255, 0, 0), 1)
        cv2.putText(tmp, f"{box[4]:.2f}", (box[:2] / RATIO).astype(int) + [0, 15], cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    for p in detections:
        cv2.circle(tmp, (p[-2:] // RATIO).astype(int), 2, (255, 0, 0), 3)
    

    # plt.figure(figsize=(8, 8))
    # plt.imshow(tmp)
    return tmp

In [5]:
vpaths = glob.glob("/home/kyamkovyi/Videos/*.avi")
len(vpaths)

13

In [6]:
for vpath in tqdm(vpaths[:1]):
    frames = []

    cap = cv2.VideoCapture(vpath)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
        frames[-1] = cv2.cvtColor(frames[-1], cv2.COLOR_BGR2RGB)

    break
    results = []
    for frame in frames:
        dets = inferer.infer(frame)
        res = draw(frame, dets)
        results.append(res)

    cap = cv2.VideoWriter(vpath.replace("/home/kyamkovyi/Videos/", "/home/kyamkovyi/Videos/results/"), cv2.VideoWriter_fourcc(*'DIVX'), 1, (640, 480))
    for res in results:
        cap.write(cv2.cvtColor(res, cv2.COLOR_RGB2BGR))

  0%|          | 0/1 [00:00<?, ?it/s]


In [30]:
with open("../gw_prototype/camera_calibration/params/pg2-video_20241119142311.json", "rt") as f:
    data = json.load(f)

K = np.array([[250, 0, 320], [0, 250, 240], [0, 0, 1]])  # Example intrinsic matrix
D = np.array([-0.3, 0.1, 0, 0])  # Example distortion coefficients

K = np.array([
    [data["LENS_PARAMS"]["FX"], 0, data["LENS_PARAMS"]["CX"]],
    [0, data["LENS_PARAMS"]["FY"], data["LENS_PARAMS"]["CY"]],
    [0, 0, 1]
])

# Apply fisheye correction
def apply_undistortion(image):
    DIM = (image.shape[1], image.shape[0])  # Image dimensions
    K = np.array([[250, 0, DIM[0]//2], [0, 250, DIM[1]//2], [0, 0, 1]])  # Example intrinsic matrix
    D = np.array([-0.3, 0.1, 0, 0])  # Example distortion coefficients
    
    # Generate new camera matrix
    map1, map2 = cv2.initUndistortRectifyMap(K, D, None, K, DIM, cv2.CV_16SC2)
    undistorted_image = cv2.remap(image, map1, map2, interpolation=cv2.INTER_LINEAR)
    
    return undistorted_image

def apply_fisheye_undistortion(image):
    DIM = (image.shape[1], image.shape[0])  # Image dimensions

    D = np.array([data["LENS_PARAMS"]["K0"], data["LENS_PARAMS"]["K1"], data["LENS_PARAMS"]["K2"], data["LENS_PARAMS"]["K3"]])
    # Generate new camera matrix
    map1, map2 = cv2.fisheye.initUndistortRectifyMap(K, D, None, K, DIM, cv2.CV_16SC2)
    undistorted_image = cv2.remap(image, map1, map2, interpolation=cv2.INTER_LINEAR)

    return undistorted_image

# Apply sharpening filter to enhance edges
def apply_correction(image):
    sharpen_kernel = np.array([[0, -1, 0],
                            [-1, 5, -1],
                            [0, -1, 0]])
    sharpened_image = cv2.filter2D(image, -1, sharpen_kernel)
    
    # Improve contrast using histogram equalization (adaptive for better results)
    lab = cv2.cvtColor(sharpened_image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    enhanced_image = cv2.merge((l, a, b))
    enhanced_image = cv2.cvtColor(enhanced_image, cv2.COLOR_LAB2BGR)
    
    return enhanced_image

In [38]:
res_path = "/home/kyamkovyi/Videos/results_4/"

shutil.rmtree(res_path, ignore_errors=True)
os.mkdir(res_path)

for vpath in tqdm(vpaths):
    frames = []

    cap = cv2.VideoCapture(vpath)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
        frames[-1] = cv2.cvtColor(frames[-1], cv2.COLOR_BGR2RGB)

    results = []
    for frame in frames:
        full_frame = np.zeros((480, 640 * 3, 3), dtype=np.uint8)
        dets = inferer.infer(frame)
        res = draw(frame, dets)
        full_frame[:, :640] = res
        
        unistort_img = apply_undistortion(frame)
        dets = inferer.infer(unistort_img)
        res = draw(unistort_img, dets)
        full_frame[:, 640:1280] = res
        
        corrected_img = apply_correction(unistort_img)
        dets = inferer.infer(corrected_img)
        res = draw(corrected_img, dets)
        full_frame[:, 1280:] = res
        
        results.append(full_frame)

    cap = cv2.VideoWriter(vpath.replace("/home/kyamkovyi/Videos/", res_path), cv2.VideoWriter_fourcc(*'DIVX'), 1, (640 * 3, 480))
    for res in results:
        cap.write(cv2.cvtColor(res, cv2.COLOR_RGB2BGR))
    cap.release()

  0%|          | 0/13 [00:00<?, ?it/s]

100%|██████████| 13/13 [01:19<00:00,  6.13s/it]
